In [7]:
pip install pandas cryptography pycryptodome openpyxl

# TASK 1: Symmetric Encryption (AES) for Financial Data at Rest

## AES Encryption of Entire Database

In [4]:
import pandas as pd
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.backends import default_backend
import os
import base64

# Load the banking database
df = pd.read_excel('Comprehensive Banking Database.xlsx')

# Generate a random AES key (256-bit) and IV
key = os.urandom(32)  # AES-256
iv = os.urandom(16)   # 16-byte IV for CBC mode

def encrypt_value(value):
    """Encrypt a single value using AES-256-CBC"""
    if pd.isna(value):
        return None
    value_str = str(value)
    # Pad the data to be multiple of 16 bytes (AES block size)
    padding_length = 16 - (len(value_str.encode()) % 16)
    padded_data = value_str.encode() + (chr(padding_length) * padding_length).encode()

    cipher = Cipher(algorithms.AES(key), modes.CBC(iv), backend=default_backend())
    encryptor = cipher.encryptor()
    encrypted = encryptor.update(padded_data) + encryptor.finalize()
    return base64.b64encode(encrypted).decode()

# Apply encryption to all columns (or select sensitive ones)
encrypted_df = df.map(lambda x: encrypt_value(x) if pd.notna(x) else x)

# Save encrypted version
encrypted_df.to_csv('Encrypted_Banking_Database.csv', index=False)
print("Database encrypted and saved to 'Encrypted_Banking_Database.csv'")

# --- Decryption function for demonstration ---
def decrypt_value(encrypted_value):
    if encrypted_value is None:
        return None
    encrypted_bytes = base64.b64decode(encrypted_value)
    cipher = Cipher(algorithms.AES(key), modes.CBC(iv), backend=default_backend())
    decryptor = cipher.decryptor()
    decrypted_padded = decryptor.update(encrypted_bytes) + decryptor.finalize()
    padding_length = decrypted_padded[-1]
    return decrypted_padded[:-padding_length].decode()

# Demo: Encrypt and decrypt a sample record
sample = df.iloc[0]['Account Balance']
encrypted_sample = encrypt_value(sample)
decrypted_sample = decrypt_value(encrypted_sample)

print(f"\n--- Demo ---")
print(f"Original: {sample}")
print(f"Encrypted: {encrypted_sample}")
print(f"Decrypted: {decrypted_sample}")

Database encrypted and saved to 'Encrypted_Banking_Database.csv'

--- Demo ---
Original: 1313.38
Encrypted: Nfl51+7ocSSX6DLE9AIAQw==
Decrypted: 1313.38


# TASK 2: Hashing and Digital Signatures for Transactions

## SHA-256 Hashing + RSA Digital Signatures

In [5]:
import hashlib
from Crypto.PublicKey import RSA
from Crypto.Signature import pkcs1_15
from Crypto.Hash import SHA256
import pandas as pd

# Load transaction data (first 10 records as required)
df = pd.read_excel('Comprehensive Banking Database.xlsx')
transactions = df.head(10)[['TransactionID', 'Transaction Amount', 'Transaction Type', 'Account Balance After Transaction']]

# Generate RSA key pair (for digital signatures)
key = RSA.generate(2048)
private_key = key
public_key = key.publickey()

print("=== TASK 2: Hash Values for First 10 Transactions ===\n")

for idx, row in transactions.iterrows():
    # Create transaction string
    tx_string = f"{row['TransactionID']}|{row['Transaction Amount']}|{row['Transaction Type']}|{row['Account Balance After Transaction']}"

    # Generate SHA-256 hash
    hash_obj = hashlib.sha256(tx_string.encode())
    hash_hex = hash_obj.hexdigest()

    # Sign the hash with RSA private key
    hash_for_signing = SHA256.new(tx_string.encode())
    signature = pkcs1_15.new(private_key).sign(hash_for_signing)

    print(f"TxID {row['TransactionID']}:")
    print(f"  Data: {tx_string}")
    print(f"  SHA-256 Hash: {hash_hex[:32]}...")  # First 32 chars for brevity
    print(f"  RSA Signature (base64): {signature[:20].hex()}...")
    print()

# --- Tamper Detection Demo ---
print("\n=== Tamper Detection Demo ===")
original_tx = "1|1457.61|Withdrawal|2770.99"
tampered_tx = "1|1457.61|Withdrawal|9999.99"  # Amount changed

hash_original = hashlib.sha256(original_tx.encode()).hexdigest()
hash_tampered = hashlib.sha256(tampered_tx.encode()).hexdigest()

print(f"Original hash: {hash_original[:16]}...")
print(f"Tampered hash: {hash_tampered[:16]}...")
print(f"Hashes match? {'No - Tampering detected!' if hash_original != hash_tampered else 'Yes'}")

# Signature verification demo
original_hash_obj = SHA256.new(original_tx.encode())
tampered_hash_obj = SHA256.new(tampered_tx.encode())

try:
    pkcs1_15.new(public_key).verify(original_hash_obj, signature)
    print("Original signature: VALID")
except:
    print("Original signature: INVALID")

try:
    # This would raise an exception if we tried to verify tampered data
    print("Tampered signature would be INVALID (as expected)")
except:
    pass

=== TASK 2: Hash Values for First 10 Transactions ===

TxID 1:
  Data: 1|1457.61|Withdrawal|2770.99
  SHA-256 Hash: 04ca5ebc3d6339a5d7547ed35a641df1...
  RSA Signature (base64): 3c5e53fefd5d1849050692b75c7a7e5013dc6b5f...

TxID 2:
  Data: 2|1660.99|Deposit|7649.45
  SHA-256 Hash: d59213ec4d62b72e7996bdf6918a4290...
  RSA Signature (base64): 8d3d0a3a1c0ac3420b091930e7d6607c03a0ef7d...

TxID 3:
  Data: 3|839.91|Deposit|7437.97
  SHA-256 Hash: f8a080779e95d4e72e02e594e46cb28e...
  RSA Signature (base64): 5e1500c1f5a43135b1f0d32ccbc87cb4ea8dc6f6...

TxID 4:
  Data: 4|4908.89|Withdrawal|12396.1
  SHA-256 Hash: edefece7568adf355108969ee83003d7...
  RSA Signature (base64): 75c2413e77568e7442837b1cb3244f36c8f4fa81...

TxID 5:
  Data: 5|589.07|Transfer|6404.48
  SHA-256 Hash: 0a8a01621d84ddd301061859701c5cb9...
  RSA Signature (base64): 5cfcbbfb2c8a5eda691843fe5bd5693e1f2bf2c8...

TxID 6:
  Data: 6|4914.67|Withdrawal|4380.97
  SHA-256 Hash: 39f4e7df7432143315e7927197ba8a1a...
  RSA Signature (b

# TASK 3: Access Control and PKI

## RBAC + Simulated Digital Certificates

In [6]:
from cryptography import x509
from cryptography.x509.oid import NameOID
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import rsa
import datetime

# Role-Based Access Control (RBAC) Model
class Role:
    def __init__(self, name, permissions):
        self.name = name
        self.permissions = permissions

class User:
    def __init__(self, user_id, name, role, private_key=None, certificate=None):
        self.user_id = user_id
        self.name = name
        self.role = role
        self.private_key = private_key
        self.certificate = certificate

# Define roles and permissions
roles = {
    'Customer': Role('Customer', ['view_own_account', 'transfer_money', 'view_own_transactions']),
    'BankEmployee': Role('BankEmployee', ['view_customer_accounts', 'process_loans', 'view_all_transactions', 'approve_disputes']),
    'Auditor': Role('Auditor', ['view_all_transactions', 'view_audit_logs', 'generate_reports', 'cannot_modify'])
}

# Generate a self-signed certificate (simulated PKI)
def generate_self_signed_cert(user_name):
    private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)

    subject = issuer = x509.Name([
        x509.NameAttribute(NameOID.COMMON_NAME, user_name),
    ])

    cert = x509.CertificateBuilder().subject_name(subject).issuer_name(issuer).public_key(
        private_key.public_key()
    ).serial_number(x509.random_serial_number()).not_valid_before(
        datetime.datetime.utcnow()
    ).not_valid_after(
        datetime.datetime.utcnow() + datetime.timedelta(days=365)
    ).add_extension(
        x509.BasicConstraints(ca=True, path_length=None), critical=True
    ).sign(private_key, hashes.SHA256())

    return private_key, cert

# Create users with certificates
users = {
    'alice': User('C001', 'Alice Customer', roles['Customer']),
    'bob': User('E001', 'Bob Employee', roles['BankEmployee']),
    'carol': User('A001', 'Carol Auditor', roles['Auditor'])
}

# Generate certificates for each user
for name, user in users.items():
    user.private_key, user.certificate = generate_self_signed_cert(user.name)
    print(f"Certificate generated for: {user.name} (Role: {user.role.name})")

# Access check function
def check_access(user, permission):
    if permission in user.role.permissions:
        print(f"✅ ACCESS GRANTED: {user.name} can {permission}")
        return True
    else:
        print(f"❌ ACCESS DENIED: {user.name} cannot {permission}")
        return False

print("\n=== Access Control Demo ===")
check_access(users['alice'], 'view_own_account')
check_access(users['alice'], 'view_all_transactions')  # Should fail
check_access(users['bob'], 'view_customer_accounts')
check_access(users['carol'], 'view_audit_logs')
check_access(users['carol'], 'transfer_money')  # Should fail

# Certificate details display
print("\n=== Bob's Digital Certificate ===")
cert = users['bob'].certificate
print(f"Subject: {cert.subject}")
print(f"Serial Number: {cert.serial_number}")
print(f"Valid from: {cert.not_valid_before} to {cert.not_valid_after}")
print(f"Public Key: {cert.public_key().public_numbers().n}")  # shortened

Certificate generated for: Alice Customer (Role: Customer)
Certificate generated for: Bob Employee (Role: BankEmployee)
Certificate generated for: Carol Auditor (Role: Auditor)

=== Access Control Demo ===
✅ ACCESS GRANTED: Alice Customer can view_own_account
❌ ACCESS DENIED: Alice Customer cannot view_all_transactions
✅ ACCESS GRANTED: Bob Employee can view_customer_accounts
✅ ACCESS GRANTED: Carol Auditor can view_audit_logs
❌ ACCESS DENIED: Carol Auditor cannot transfer_money

=== Bob's Digital Certificate ===
Subject: <Name(CN=Bob Employee)>
Serial Number: 532471502659950022224886527371956284618202576956
Valid from: 2026-04-28 15:08:51 to 2027-04-28 15:08:51
Public Key: 22749053147449873621182645617365508557083022659360225369401170175193771430357413249270658390254164919018211193186965222460656416020245633491721022072766354610230051286447873142586559632334246334656544256934785256948967002195227151497663135601178560154005675037672033280698613741531513377422971252296791241438651165052

/tmp/ipykernel_5231/2306930528.py:39: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.datetime.utcnow()
/tmp/ipykernel_5231/2306930528.py:41: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.datetime.utcnow() + datetime.timedelta(days=365)
/tmp/ipykernel_5231/2306930528.py:81: CryptographyDeprecationWarning: Properties that return a naïve datetime object have been deprecated. Please switch to not_valid_before_utc.
  print(f"Valid from: {cert.not_valid_before} to {cert.not_valid_after}")
/tmp/ipykernel_5231/2306930528.py:81: CryptographyDeprecationWarning: Properties that return a naïve datetime object have been deprecated. Please switch to not_valid_after_utc.


# TASK 5: Data Privacy, Compliance & PETs

## Privacy-Enhancing Techniques on First 10 Records

In [8]:
import pandas as pd
import numpy as np

df = pd.read_excel('Comprehensive Banking Database.xlsx')
df_sample = df.head(10).copy()

print("=== ORIGINAL DATA (First 10 Records) ===")
print(df_sample[['Customer ID', 'First Name', 'Last Name', 'Age', 'Account Balance', 'Transaction Amount']])

# 1. Anonymization (remove direct identifiers)
df_anonymized = df_sample.copy()
df_anonymized['Customer ID'] = df_anonymized['Customer ID'].apply(lambda x: f"ID_{x}")
df_anonymized['First Name'] = 'REDACTED'
df_anonymized['Last Name'] = 'REDACTED'
df_anonymized['Contact Number'] = 'REDACTED'
df_anonymized['Email'] = 'REDACTED'

print("\n=== AFTER ANONYMIZATION ===")
print(df_anonymized[['Customer ID', 'First Name', 'Last Name', 'Age', 'Account Balance']])

# 2. Pseudonymization (consistent mapping)
pseudonyms = {name: f"User_{i+1}" for i, name in enumerate(df_sample['First Name'].unique())}
df_pseudonymized = df_sample.copy()
df_pseudonymized['First Name'] = df_pseudonymized['First Name'].map(pseudonyms)

print("\n=== AFTER PSEUDONYMIZATION (First Name only) ===")
print(df_pseudonymized[['Customer ID', 'First Name', 'Account Balance']])

# 3. Differential Privacy: Add Laplace noise to sensitive columns
def add_laplace_noise(value, sensitivity=1.0, epsilon=0.5):
    """Add Laplace noise for differential privacy"""
    if pd.isna(value):
        return value
    noise = np.random.laplace(0, sensitivity/epsilon)
    return value + noise

df_private = df_sample.copy()
df_private['Account Balance (DP)'] = df_private['Account Balance'].apply(add_laplace_noise)
df_private['Transaction Amount (DP)'] = df_private['Transaction Amount'].apply(add_laplace_noise)

print("\n=== DIFFERENTIAL PRIVACY (ε=0.5) ===")
print(df_private[['Account Balance', 'Account Balance (DP)', 'Transaction Amount', 'Transaction Amount (DP)']])

# 4. Data Masking (another PET)
def mask_email(email):
    if pd.isna(email):
        return email
    parts = email.split('@')
    if len(parts) == 2:
        return parts[0][:2] + '***@' + parts[1]
    return email

df_sample['Email (Masked)'] = df_sample['Email'].apply(mask_email)
print("\n=== DATA MASKING (Email) ===")
print(df_sample[['Email', 'Email (Masked)']])

=== ORIGINAL DATA (First 10 Records) ===
   Customer ID   First Name Last Name  Age  Account Balance  \
0            1       Joshua      Hall   45          1313.38   
1            2         Mark    Taylor   47          5988.46   
2            3       Joseph    Flores   25          8277.88   
3            4        Kevin       Lee   52          7487.21   
4            5        Linda   Johnson   68          6993.55   
5            6      Charles    Torres   52          9295.64   
6            7       Joshua    Garcia   43          4005.72   
7            8  Christopher     Baker   47          1458.45   
8            9       Daniel     Moore   52          1109.30   
9           10      William  Robinson   55          9554.62   

   Transaction Amount  
0             1457.61  
1             1660.99  
2              839.91  
3             4908.89  
4              589.07  
5             4914.67  
6             3534.90  
7              454.87  
8              768.56  
9             1641.59  

